# Telco Churn: Naive Bayes con tuning y validación cruzada

Se carga el conjunto **IBM Telco Customer Churn** desde una URL pública, se preprocesan variables numéricas y categóricas, y se ajusta un **GaussianNB** con búsqueda de hiperparámetros (`GridSearchCV`) y evaluación por **validación cruzada estratificada**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna().copy()

y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print(f"Filas: {X.shape[0]}, features: {X.shape[1]}")
print(f"Churn rate: {y.mean():.2%}")

Filas: 7032, features: 19
Churn rate: 26.58%


## Pipeline y búsqueda de hiperparámetros

- **Preprocesado:** `StandardScaler` en numéricas; `OneHotEncoder` en categóricas (matriz densa para `GaussianNB`).
- **Modelo:** `GaussianNB`; se tunnea `var_smoothing` (estabilidad numérica de las varianzas por clase).
- **CV:** `StratifiedKFold` para mantener la proporción de churn en cada fold.

In [ ]:
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols,
        ),
    ]
)

pipe = Pipeline(
    [
        ("prep", preprocess),
        ("clf", GaussianNB()),
    ]
)

param_grid = {
    "clf__var_smoothing": np.logspace(-12, -6, num=13),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

search.fit(X, y)

print("Mejores hiperparámetros:", search.best_params_)
print(f"Mejor ROC-AUC (CV): {search.best_score_:.4f}")

Mejores hiperparámetros: {'clf__var_smoothing': np.float64(3.162277660168379e-07)}
Mejor ROC-AUC (CV): 0.8175


## Validación cruzada del mejor modelo

Se recalcula la puntuación con `cross_val_score` sobre el **pipeline ya calibrado** (mismo esquema de CV) para reportar media y desviación típica.

In [ ]:
best_model = search.best_estimator_

cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)

print("ROC-AUC por fold:", np.round(cv_scores, 4))
print(f"Media: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

ROC-AUC por fold: [0.8273 0.8188 0.815  0.8112 0.8153]
Media: 0.8175 (+/- 0.0054)
